In [3]:
pip install kaggle

Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import numpy as np
import os
import glob

DATA_DIR = "acc102/data"
OUTPUT_DIR = "acc102/output"

def download_kaggle_dataset(dataset_name, save_path=DATA_DIR):
    cmd = f'kaggle datasets download -d {dataset_name} -p "{save_path}" --unzip'
    result = os.system(cmd)

    if result == 0:
        print(f"Download completed: {dataset_name}")
    else:
        raise RuntimeError(
            "Kaggle download failed. Please check:\n"
            "1. kaggle package is installed\n"
            "2. kaggle.json is configured\n"
            "3. dataset name is correct"
        )

def find_data_file(data_dir=DATA_DIR):
    csv_files = glob.glob(os.path.join(data_dir, "*.csv"))
    xlsx_files = glob.glob(os.path.join(data_dir, "*.xlsx"))
    all_files = csv_files + xlsx_files

    if not all_files:
        raise FileNotFoundError(f"No CSV/XLSX file found in {data_dir}")

    print("Found data file:", all_files[0])
    return all_files[0]

def load_data(file_path):
    if file_path.lower().endswith(".csv"):
        return pd.read_csv(file_path, encoding="latin1")
    elif file_path.lower().endswith(".xlsx"):
        return pd.read_excel(file_path)
    else:
        raise ValueError("Only CSV and XLSX files are supported.")

COLUMN_ALIASES = {
    "customer_id": ["CustomerID", "Customer ID", "UserID", "User ID", "customer_id"],
    "order_id": ["InvoiceNo", "Order ID", "OrderID", "Transaction ID", "invoice_no"],
    "order_date": ["InvoiceDate", "Order Date", "Date", "Transaction Date", "order_date"],
    "quantity": ["Quantity", "Qty", "quantity"],
    "unit_price": ["UnitPrice", "Price", "Unit Price", "unit_price"],
    "sales": ["Sales", "Revenue", "Amount", "TotalAmount", "sales"],
    "country": ["Country", "country"],
    "product_name": ["Description", "Product Name", "Product", "product_name"]
}


def standardize_columns(df):
    rename_map = {}
    cols = list(df.columns)

    for std_col, aliases in COLUMN_ALIASES.items():
        for a in aliases:
            if a in cols:
                rename_map[a] = std_col
                break

    df = df.rename(columns=rename_map)
    return df

def ensure_sales_column(df):
    if "sales" not in df.columns:
        if "quantity" in df.columns and "unit_price" in df.columns:
            df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
            df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
            df["sales"] = df["quantity"] * df["unit_price"]
        else:
            raise ValueError("Dataset must contain sales or quantity + unit_price.")
    else:
        df["sales"] = pd.to_numeric(df["sales"], errors="coerce")

    return df

def clean_data(df):
    df = standardize_columns(df)

    required = ["customer_id", "order_id", "order_date"]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    df = ensure_sales_column(df)

    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    df["sales"] = pd.to_numeric(df["sales"], errors="coerce")

    df = df.dropna(subset=["customer_id", "order_id", "order_date", "sales"])
    df = df[df["sales"].notna()]

    df["year_month"] = df["order_date"].dt.to_period("M").astype(str)

    return df

def mark_cancelled_orders(df):
    df = df.copy()
    df["is_cancelled"] = df["order_id"].astype(str).str.upper().str.startswith("C")
    return df

def build_user_month_features(df):
    df = mark_cancelled_orders(df)

    order_level = df.groupby(["customer_id", "year_month", "order_id"], as_index=False).agg(
        order_sales=("sales", "sum"),
        is_cancelled=("is_cancelled", "max")
    )

    user_month = order_level.groupby(["customer_id", "year_month"], as_index=False).agg(
        monthly_expense=("order_sales", "sum"),
        monthly_purchase_frequency=("order_id", "nunique"),
        cancelled_orders=("is_cancelled", "sum")
    )

    user_month["cancel_rate"] = np.where(
        user_month["monthly_purchase_frequency"] > 0,
        user_month["cancelled_orders"] / user_month["monthly_purchase_frequency"],
        0
    )
    user_month["cancel_rate_pct"] = (user_month["cancel_rate"] * 100).round(2)

    return user_month

def identify_high_frequency_users(user_month_df, min_monthly_orders=5):
    return user_month_df[user_month_df["monthly_purchase_frequency"] >= min_monthly_orders].copy()

def add_income_proxy(df):
    df = df.copy()

    conditions = [
        df["monthly_expense"] < 1000,
        (df["monthly_expense"] >= 1000) & (df["monthly_expense"] < 3000),
        (df["monthly_expense"] >= 3000) & (df["monthly_expense"] < 8000),
        df["monthly_expense"] >= 8000
    ]
    levels = ["Low", "Lower-Middle", "Middle", "High"]

    df["income_proxy_level"] = np.select(conditions, levels, default="Unknown")
    df["estimated_monthly_income"] = (df["monthly_expense"] / 0.35).round(2)

    return df

def save_output(df, output_path=f"{OUTPUT_DIR}/high_frequency_user_profiles.csv"):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Saved to: {output_path}")

def main():
    dataset_name = "carrie1/ecommerce-data"

    existing_files = glob.glob(os.path.join(DATA_DIR, "*.csv")) + glob.glob(os.path.join(DATA_DIR, "*.xlsx"))
    if not existing_files:
        print("No local data file found. Downloading from Kaggle...")
        download_kaggle_dataset(dataset_name, DATA_DIR)
    else:
        print("Data file already exists. Skip download.")
    file_path = find_data_file(DATA_DIR)

    df = load_data(file_path)
    print("Raw shape:", df.shape)

    df = clean_data(df)
    print("Cleaned shape:", df.shape)

    user_month_df = build_user_month_features(df)
    print("User-month feature shape:", user_month_df.shape)

    high_freq_df = identify_high_frequency_users(user_month_df, min_monthly_orders=5)
    print("High-frequency user shape:", high_freq_df.shape)

    high_freq_df = add_income_proxy(high_freq_df)

    save_output(high_freq_df)

    print("\nSample output:")
    print(high_freq_df.head())


if __name__ == "__main__":
    main()

No local data file found. Downloading from Kaggle...
Download completed: carrie1/ecommerce-data
Found data file: acc102/data\data.csv
Raw shape: (541909, 8)
Cleaned shape: (406829, 10)
User-month feature shape: (13675, 7)
High-frequency user shape: (416, 7)
Saved to: acc102/output/high_frequency_user_profiles.csv

Sample output:
     customer_id year_month  monthly_expense  monthly_purchase_frequency  \
15       12352.0    2011-03           304.68                           7   
136      12409.0    2011-09          4854.80                           5   
157      12415.0    2011-11          5918.15                           6   
191      12428.0    2011-03          5924.62                           6   
270      12457.0    2011-04           842.35                           5   

     cancelled_orders  cancel_rate  cancel_rate_pct income_proxy_level  \
15                  3     0.428571            42.86                Low   
136                 3     0.600000            60.00             